In [1]:
!pip install -q datasets huggingface_hub

from importlib.metadata import version

pkgs = [
    "tokenizers",
    "torch",
    "datasets",
]
for p in pkgs:
    print(f"{p} version: {version(p)}")


tokenizers version: 0.22.2
torch version: 2.11.0+cu128
datasets version: 4.0.0


# Hybrid: Training-From-Scratch Architecture

This notebook implements a 100M-class transformer with the **Hybrid tokenizer** (Qwen BPE + AGROVOC entity injection, vocab from file).

We define the following components:
- **SwiGLU FeedForward**
- **RMSNorm** (no bias, float32 stable)
- **RoPE** (rotary positional encodings)
- **Grouped Query Attention** (GQA)
- **Transformer Block** (pre-norm, residual)
- **Factorized Embedding** (low-rank input projection)
- **Qwen3Model** (full transformer with tied factorized output)

## Architecture Code

In [ ]:
import torch
import torch.nn as nn


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], bias=False)

        # ======================================
        # Runtime Metrics
        # ======================================
        self.latest_metrics = {
            "mlp_output_norm": 0.0,
            "hidden_variance": 0.0,
            "hidden_mean": 0.0,
            "hidden_max": 0.0,
            "hidden_min": 0.0,
            "hidden_norm": 0.0,
            "dead_neurons": 0.0,
            "activation_sparsity": 0.0,
        }

        self.alert_flags = {
            "nan_detected": False,
            "activation_explosion": False,
            "collapse_detected": False,
        }

        self.forward_calls = 0

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        # ======================================
        # Count Forward Passes
        # ======================================
        self.forward_calls += 1

        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)

        hidden = nn.functional.silu(x_fc1) * x_fc2

        output = self.fc3(hidden)

        # ======================================
        # Numerical Stability
        # ======================================
        if not torch.isfinite(output).all():
            raise RuntimeError(
                "FeedForward produced NaN or Inf values."
            )

        with torch.no_grad():

            # Cache repeated computations
            output_norm = output.norm(p=2)
            hidden_variance = hidden.var()

            self.latest_metrics.update({

                "mlp_output_norm": output_norm.item(),

                "hidden_variance": hidden_variance.item(),

                "hidden_mean": hidden.mean().item(),

                "hidden_max": hidden.max().item(),

                "hidden_min": hidden.min().item(),

                "hidden_norm": hidden.norm(p=2).item(),

                "dead_neurons": (
                    (hidden.abs() < 1e-8)
                    .float()
                    .mean()
                    .item()
                ),

                "activation_sparsity": (
                    (hidden.abs() < 1e-4)
                    .float()
                    .mean()
                    .item()
                ),
            })

            self.alert_flags["nan_detected"] = (
                torch.isnan(output).any().item()
            )

            self.alert_flags["activation_explosion"] = (
                output_norm > 1e4
            )

            self.alert_flags["collapse_detected"] = (
                hidden_variance < 1e-8
            )

        return output

In [ ]:
import torch
import torch.nn as nn


class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6, qwen3_compatible=True):
        super().__init__()

        self.eps = eps
        self.qwen3_compatible = qwen3_compatible

        self.scale = nn.Parameter(torch.ones(emb_dim))

        # =====================================================
        # NEW: Layer Metadata
        # =====================================================
        self.layer_name = "RMSNorm"
        self.forward_calls = 0
        self.num_parameters = sum(p.numel() for p in self.parameters())

        # =====================================================
        # MODIFIED: Expanded Debug Metrics Buffer
        # Used by CallbackManager / MLflow
        # =====================================================
        self.latest_metrics = {
            "variance": 0.0,
            "norm": 0.0,

            # ---------- NEW ----------
            "output_mean": 0.0,
            "output_std": 0.0,
            "output_max": 0.0,
            "output_min": 0.0,
            "scale_mean": 0.0,
            "scale_std": 0.0,
        }

        # =====================================================
        # NEW: Alert Flags
        # =====================================================
        self.alert_flags = {
            "nan_detected": False,
            "variance_collapse": False,
            "activation_explosion": False,
        }

    # =====================================================
    # MODIFIED: Type Hint
    # =====================================================
    def forward(self, x: torch.Tensor) -> torch.Tensor:

        # ---------- NEW ----------
        self.forward_calls += 1

        input_dtype = x.dtype

        if self.qwen3_compatible:
            x = x.to(torch.float32)

        variance = x.pow(2).mean(dim=-1, keepdim=True)

        norm_x = x * torch.rsqrt(variance + self.eps)

        output = norm_x * self.scale

        # =====================================================
        # NEW: Numerical Stability Check
        # =====================================================
        if not torch.isfinite(output).all():
            raise RuntimeError(
                "RMSNorm produced NaN or Inf values."
            )

        # =====================================================
        # MODIFIED: Extended Monitoring Metrics
        # =====================================================
        with torch.no_grad():

            self.latest_metrics.update({

                # Existing
                "variance": variance.mean().item(),
                "norm": self.scale.norm(p=2).item(),

                # ---------- NEW ----------
                "output_mean": output.mean().item(),

                "output_std": output.std().item(),

                "output_max": output.max().item(),

                "output_min": output.min().item(),

                "scale_mean": self.scale.mean().item(),

                "scale_std": self.scale.std().item(),

            })

            # =================================================
            # NEW: Alert Flags
            # =================================================
            self.alert_flags["nan_detected"] = (
                torch.isnan(output).any().item()
            )

            self.alert_flags["variance_collapse"] = (
                variance.mean() < 1e-8
            )

            self.alert_flags["activation_explosion"] = (
                output.norm() > 1e4
            )

        return output.to(input_dtype)

In [ ]:
def compute_rope_params(
    head_dim: int,
    theta_base: float = 10_000,
    context_length: int = 4096,
    dtype: torch.dtype = torch.float32,
):
    # ==========================================================
# Computes precomputed Rotary Position Embedding (RoPE)
# cosine and sine matrices.
#
# Returns:
#     cos : (context_length, head_dim)
#     sin : (context_length, head_dim)
#
# These tensors are computed once and reused throughout
# training and inference.
# ==========================================================
    # ==========================
# NEW
# Input Validation
# ==========================

    assert head_dim % 2 == 0

    assert head_dim > 0, "head_dim must be positive."

    assert context_length > 0, "context_length must be positive."

    assert theta_base > 0, "theta_base must be positive."

    # Compute the inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float() / head_dim))

    # Generate position indices
    positions = torch.arange(context_length, dtype=dtype)

    # Compute the angles
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # Shape: (context_length, head_dim // 2)

    # Expand angles to match the head_dim
    angles = torch.cat([angles, angles], dim=1)  # Shape: (context_length, head_dim)

    # Precompute sine and cosine
    cos = torch.cos(angles)
    sin = torch.sin(angles)

        # ==========================
    # NEW
    # Numerical Stability
    # ==========================

    if not torch.isfinite(cos).all():
        raise RuntimeError("RoPE cosine contains NaN/Inf.")

    if not torch.isfinite(sin).all():
        raise RuntimeError("RoPE sine contains NaN/Inf.")

    return cos, sin


def apply_rope(
    x: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
) -> torch.Tensor:
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    # ==========================
    # NEW
    # Shape Validation
    # ==========================

    assert cos.shape[0] >= seq_len, \
        "RoPE cosine cache shorter than sequence."

    assert sin.shape[0] >= seq_len, \
        "RoPE sine cache shorter than sequence."
    assert head_dim % 2 == 0, "Head dimension must be even"

    # ==========================
    # NEW
    # Numerical Check
    # ==========================

    if not torch.isfinite(x).all():
        raise RuntimeError(
            "Input to apply_rope contains NaN/Inf."
        )

    half_dim = head_dim // 2

    # Split x into first half and second half
    x1 = x[..., : half_dim]  # First half
    x2 = x[..., half_dim:]  # Second half

    # Adjust sin and cos shapes
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, seq_len, head_dim)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # Apply the rotary transformation
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

        # ==========================
    # NEW
    # Numerical Stability
    # ==========================

    if not torch.isfinite(x_rotated).all():
        raise RuntimeError(
            "RoPE output contains NaN/Inf."
        )

    # It's ok to use lower-precision after applying cos and sin rotation
    return x_rotated.to(dtype=x.dtype)

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(
        self, d_in, num_heads, num_kv_groups, head_dim=None, qk_norm=False
    ):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        if head_dim is None:
            assert d_in % num_heads == 0, "`d_in` must be divisible by `num_heads` if `head_dim` is not set"
            head_dim = d_in // num_heads

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim

        self.W_query = nn.Linear(d_in, self.d_out, bias=False)
        self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=False)
        self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=False)

        self.out_proj = nn.Linear(self.d_out, d_in, bias=False)
        self.layer_name = "GroupedQueryAttention"
        self.forward_calls = 0
        self.num_parameters = sum(
            p.numel() for p in self.parameters()
        )

        if qk_norm:
            self.q_norm = RMSNorm(head_dim, eps=1e-6)
            self.k_norm = RMSNorm(head_dim, eps=1e-6)
        else:
            self.q_norm = self.k_norm = None

                # =====================================================
        # MODIFIED: Debug Metrics Buffer
        # =====================================================

        self.latest_metrics = {

            "attention_entropy": 0.0,

            "attention_sparsity": 0.0,

            "attention_output_norm": 0.0,

            "attention_variance": 0.0,

            # ---------- NEW ----------

            "attention_max": 0.0,

            "attention_min": 0.0,

            "attention_mean": 0.0,

            "query_norm": 0.0,

            "key_norm": 0.0,

            "value_norm": 0.0,

            "context_norm": 0.0,
        }
        self.alert_flags = {

        "nan_detected": False,

        "attention_collapse": False,

        "attention_explosion": False,

    }

    def forward(
    self,
    x: torch.Tensor,
    mask: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
):
        self.forward_calls += 1
        b, num_tokens, _ = x.shape
        if not torch.isfinite(x).all():
         raise RuntimeError(
            "Attention input contains NaN/Inf."
        )

        # Apply projections
        queries = self.W_query(x)  # (b, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)       # (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)   # (b, num_tokens, num_kv_groups * head_dim)

        # Reshape
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        # Optional normalization
        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys = self.k_norm(keys)

        # Apply RoPE
        queries = apply_rope(queries, cos, sin)
        keys = apply_rope(keys, cos, sin)

        # Expand K and V to match number of heads
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)

        # Attention
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)
        if not torch.isfinite(attn_weights).all():
         raise RuntimeError(
            "Attention weights contain NaN/Inf."
        )

        context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)
        output = self.out_proj(context)

        entropy = -(

            attn_weights *

            torch.log(attn_weights + 1e-9)

        ).sum(dim=-1).mean()

        sparsity = (

            attn_weights < 1e-3

        ).float().mean()

        output_norm = output.norm(p=2)

        output_var = output.var()

        query_norm = queries.norm(p=2)

        key_norm = keys.norm(p=2)

        value_norm = values.norm(p=2)

        context_norm = context.norm(p=2)

        with torch.no_grad():
           

            self.latest_metrics = {
                    "attention_entropy": entropy.item(),

                    "attention_sparsity": sparsity.item(),

                    "attention_output_norm": output_norm.item(),

                    "attention_variance": output_var.item(),

                    # ---------- NEW ----------

                    "attention_max": attn_weights.max().item(),

                    "attention_min": attn_weights.min().item(),

                    "attention_mean": attn_weights.mean().item(),

                    "query_norm": query_norm.item(),

                    "key_norm": key_norm.item(),

                    "value_norm": value_norm.item(),

                    "context_norm": context_norm.item(),
                    }
            self.alert_flags["nan_detected"] = (

                torch.isnan(output).any().item()

            )

            self.alert_flags["attention_explosion"] = (

                output_norm > 1e4

            )

            self.alert_flags["attention_collapse"] = (

                entropy < 0.05

            )
                        
        if not torch.isfinite(output).all():
             raise RuntimeError(
                "Attention output contains NaN/Inf."
            )
        return output
    

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            head_dim=cfg["head_dim"],
            num_kv_groups=cfg["n_kv_groups"],
            qk_norm=cfg["qk_norm"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.norm2 = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.layer_name = "TransformerBlock"

        self.forward_calls = 0

        self.num_parameters = sum(
            p.numel() for p in self.parameters()
        )


        self.latest_metrics = {

            "residual_norm_post_att": 0.0,

            "residual_norm_post_ff": 0.0,

            "residual_variance_post_att": 0.0,

            "residual_variance_post_ff": 0.0,

            # ---------- NEW ----------

            "block_output_norm": 0.0,

            "block_output_variance": 0.0,

            "shortcut_norm": 0.0,

            "shortcut_variance": 0.0,

            "residual_gain": 0.0,
        }

        self.alert_flags = {

            "nan_detected": False,

            "residual_explosion": False,

            "residual_collapse": False,

        }

    def forward(
            self,
            x: torch.Tensor,
            mask: torch.Tensor,
            cos: torch.Tensor,
            sin: torch.Tensor,
        ) -> torch.Tensor:
        self.forward_calls += 1
        if not torch.isfinite(x).all():
            raise RuntimeError(
        "TransformerBlock input contains NaN/Inf."
        )
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x, mask, cos, sin)  # Shape [batch_size, num_tokens, emb_size]
        x = x + shortcut  # Add the original input back

        res1_norm = x.norm(p=2)

        res1_var = x.var()

        
        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = x + shortcut  # Add the original input back
        res2_norm = x.norm(p=2)

        res2_var = x.var()
        shortcut_norm = shortcut.norm(p=2)

        shortcut_var = shortcut.var()

        block_output_norm = x.norm(p=2)

        block_output_var = x.var()

        residual_gain = (

            block_output_norm /

            (shortcut_norm + 1e-9)

        )

        with torch.no_grad():

            self.latest_metrics.update({
                    "residual_norm_post_att":
            res1_norm.item(),

        "residual_norm_post_ff":
            res2_norm.item(),

        "residual_variance_post_att":
            res1_var.item(),

        "residual_variance_post_ff":
            res2_var.item(),

        # ---------- NEW ----------

        "block_output_norm":
            block_output_norm.item(),

        "block_output_variance":
            block_output_var.item(),

        "shortcut_norm":
            shortcut_norm.item(),

        "shortcut_variance":
            shortcut_var.item(),

        "residual_gain":
            residual_gain.item(),
        "attention_entropy":
            self.att.latest_metrics["attention_entropy"],

        "attention_sparsity":
            self.att.latest_metrics["attention_sparsity"],

        "mlp_output_norm":
            self.ff.latest_metrics["mlp_output_norm"],

        "hidden_variance":
            self.ff.latest_metrics["hidden_variance"],
            })
        self.alert_flags["nan_detected"] = (

                torch.isnan(x).any().item()

            )

        self.alert_flags["residual_explosion"] = (

                block_output_norm > 1e4

            )

        self.alert_flags["residual_collapse"] = (

                block_output_var < 1e-8

            )
        if not torch.isfinite(x).all():
            raise RuntimeError(
        "TransformerBlock output contains NaN/Inf.")

        return x

In [ ]:
class FactorizedEmbedding(nn.Module):
    def __init__(self, vocab_size, rank, emb_dim):
        super().__init__()

        # Low-rank embedding
        self.embedding = nn.Embedding(
            vocab_size,
            rank,
        )

        # Projection to transformer dimension
        self.proj = nn.Linear(
            rank,
            emb_dim,
            bias=False,
        )
        self.layer_name = "FactorizedEmbedding"

        self.forward_calls = 0

        self.num_parameters = sum(
            p.numel() for p in self.parameters()
        )
        self.latest_metrics = {

            "embedding_norm": 0.0,

            "projection_norm": 0.0,

            "embedding_variance": 0.0,

            "projection_variance": 0.0,

            "embedding_mean": 0.0,

            "projection_mean": 0.0,
        }
        self.alert_flags = {

            "nan_detected": False,

            "embedding_collapse": False,

        }

    def forward(
            self,
            input_ids: torch.Tensor,
        ) -> torch.Tensor:
        self.forward_calls += 1
        if input_ids.min() < 0:
            raise ValueError(
                "Negative token IDs detected."
            )

        if input_ids.max() >= self.embedding.num_embeddings:
            raise ValueError(
                "Token ID exceeds vocabulary size."
            )
        embedding = self.embedding(input_ids)

        projection = self.proj(embedding)

        if not torch.isfinite(projection).all():
            raise RuntimeError(
        "FactorizedEmbedding produced NaN/Inf."
            )
        
        with torch.no_grad():

            embedding_norm = embedding.norm(p=2)

            projection_norm = projection.norm(p=2)

            embedding_var = embedding.var()

            projection_var = projection.var()

            self.latest_metrics.update({

                "embedding_norm":
                    embedding_norm.item(),

                "projection_norm":
                    projection_norm.item(),

                "embedding_variance":
                    embedding_var.item(),

                "projection_variance":
                    projection_var.item(),

                "embedding_mean":
                    embedding.mean().item(),

                "projection_mean":
                    projection.mean().item(),
                "projection_weight_norm":
                    self.proj.weight.norm(p=2).item(),

            })

            self.alert_flags["nan_detected"] = (
                torch.isnan(projection).any().item()
            )

            self.alert_flags["embedding_collapse"] = (
                embedding_var < 1e-8
            )
        return projection

In [ ]:
class Qwen3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = FactorizedEmbedding(
            vocab_size=cfg["vocab_size"],
            rank=cfg["embedding_rank"],
            emb_dim=cfg["emb_dim"],
        )

        self.trf_blocks = nn.ModuleList(
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = RMSNorm(cfg["emb_dim"])
        self.output_proj = nn.Linear(
            cfg["emb_dim"],
            cfg["embedding_rank"],
            bias=False,
        )

        if cfg["head_dim"] is None:
            head_dim = cfg["emb_dim"] // cfg["n_heads"]
        else:
            head_dim = cfg["head_dim"]
        cos, sin = compute_rope_params(
            head_dim=head_dim,
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"]
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
                # =====================================================
        # NEW: Verify RoPE Buffers
        # =====================================================
        assert self.cos.shape == self.sin.shape, \
        "RoPE cosine and sine buffers must have identical shapes."
        self.cfg = cfg
        self.model_name = "Qwen3Model"

        self.forward_calls = 0

        self.num_parameters = sum(
            p.numel() for p in self.parameters()
        )

        
        self.latest_metrics = {

            "embedding_norm": 0.0,

            "final_hidden_norm": 0.0,

            "logits_norm": 0.0,

            "logits_variance": 0.0,

            "num_tokens": 0,

            "sequence_length": 0,
        }
        self.alert_flags = {

            "nan_detected": False,

            "logit_explosion": False,

        }

    def forward(
            self,
            in_idx: torch.Tensor,
        ) -> torch.Tensor:
        self.forward_calls += 1
        if not torch.isfinite(in_idx.float()).all():
            raise RuntimeError(
                "Input tokens contain NaN/Inf."
            )

        if in_idx.max() >= self.cfg["vocab_size"]:
            raise ValueError(
                "Input token exceeds vocabulary."
            )
        tok_embeds = self.tok_emb(in_idx)
        x = tok_embeds

        num_tokens = x.shape[1]
        mask = torch.triu(torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool), diagonal=1)

        for block in self.trf_blocks:
            x = block(x, mask, self.cos, self.sin)
        x = self.final_norm(x)
        hidden = self.output_proj(x)

        logits = torch.nn.functional.linear(
            hidden,
            self.tok_emb.embedding.weight
        )
        if not torch.isfinite(logits).all():
            raise RuntimeError(
                "Model logits contain NaN/Inf."
            )
        with torch.no_grad():

                embedding_norm = tok_embeds.norm(p=2)
                hidden_norm = hidden.norm(p=2)
                logits_norm = logits.norm(p=2)
                logits_variance = logits.var()

                # =====================================================
                # Model-level Metrics
                # =====================================================
                self.latest_metrics.update({

                    "embedding_norm":
                        embedding_norm.item(),

                    "final_hidden_norm":
                        hidden_norm.item(),

                    "logits_norm":
                        logits_norm.item(),

                    "logits_variance":
                        logits_variance.item(),

                    "num_tokens":
                        in_idx.numel(),

                    "sequence_length":
                        num_tokens,
                })

                # =====================================================
                # INSERT THIS ENTIRE BLOCK HERE
                # =====================================================

                attention_entropy = []
                attention_sparsity = []
                residual_gain = []

                for block in self.trf_blocks:

                    attention_entropy.append(
                        block.att.latest_metrics["attention_entropy"]
                    )

                    attention_sparsity.append(
                        block.att.latest_metrics["attention_sparsity"]
                    )

                    residual_gain.append(
                        block.latest_metrics["residual_gain"]
                    )


                mlp_norm = []

                for block in self.trf_blocks:

                    mlp_norm.append(
                        block.ff.latest_metrics["mlp_output_norm"]
                    )
                self.latest_metrics.update({

                    "mean_attention_entropy":
                        sum(attention_entropy) / len(attention_entropy),

                    "mean_attention_sparsity":
                        sum(attention_sparsity) / len(attention_sparsity),

                    "mean_residual_gain":
                        sum(residual_gain) / len(residual_gain),
                    "output_projection_norm":
                        self.output_proj.weight.norm(p=2).item(),
                    "final_norm_variance":
                        self.final_norm.latest_metrics["variance"],
                    "mean_mlp_output_norm":
                         sum(mlp_norm) / len(mlp_norm)
                    

                })

                # =====================================================
                # KEEP YOUR EXISTING ALERTS BELOW
                # =====================================================

                self.alert_flags["nan_detected"] = any(

                    block.alert_flags["nan_detected"]

                    for block in self.trf_blocks

                )

                self.alert_flags["attention_collapse"] = any(

                    block.att.alert_flags["attention_collapse"]

                    for block in self.trf_blocks

                )

                self.alert_flags["residual_collapse"] = any(

                    block.alert_flags["residual_collapse"]

                    for block in self.trf_blocks

                )

                self.alert_flags["logit_explosion"] = (
                    logits_norm > 1e5
                )

                self.latest_hidden = hidden.detach()
                self.latest_logits = logits.detach()    
            
        return logits

## Initialize model (Hybrid, vocab=32k)

In [9]:
QWEN3_CONFIG = {
    "vocab_size": 40_000,
    "embedding_rank":256,
    "context_length": 8192,
    "emb_dim": 640,
    "n_heads": 10,
    "n_layers": 20,
    "hidden_dim": 2560,
    "head_dim": 64,
    "qk_norm": True,
    "n_kv_groups": 5,
    "rope_base": 1_000_000.0,
}

In [10]:
torch.manual_seed(123)
model = Qwen3Model(QWEN3_CONFIG)

In [ ]:


_ = model(torch.tensor([1, 2, 3]).unsqueeze(0))

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Total number of parameters: {total_params:,}")



Total number of parameters: 133,476,480


In [12]:
def estimate_model_memory(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # Calculate total number of elements per parameter
        param_size = param.numel()
        total_params += param_size
        # Check if gradients are stored for this parameter
        if param.requires_grad:
            total_grads += param_size

    # Calculate buffer size (non-parameters that require memory)
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # Size in bytes = (Number of elements) * (Size of each element in bytes)
    # We assume parameters and gradients are stored in the same type as input dtype
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # Convert bytes to gigabytes
    total_memory_gb = total_memory_bytes / (1024**3)

    return {
    "parameters": total_params,
    "gradients": total_grads,
    "buffers": total_buffers,
    "memory_gb": total_memory_gb,
    }

In [13]:
fp32_stats = estimate_model_memory(model, torch.float32)
bf16_stats = estimate_model_memory(model, torch.bfloat16)

print("FP32")
print(f"Parameters : {fp32_stats['parameters']:,}")
print(f"Gradients  : {fp32_stats['gradients']:,}")
print(f"Buffers    : {fp32_stats['buffers']:,}")
print(f"Memory     : {fp32_stats['memory_gb']:.2f} GB")

print("\nBF16")
print(f"Parameters : {bf16_stats['parameters']:,}")
print(f"Gradients  : {bf16_stats['gradients']:,}")
print(f"Buffers    : {bf16_stats['buffers']:,}")
print(f"Memory     : {bf16_stats['memory_gb']:.2f} GB")

FP32
Parameters : 133,476,480
Gradients  : 133,476,480
Buffers    : 1,048,576
Memory     : 1.00 GB

BF16
Parameters : 133,476,480
Gradients  : 133,476,480
Buffers    : 1,048,576
Memory     : 0.50 GB


In [14]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(device)
model.to(device)

cuda


Qwen3Model(
  (tok_emb): FactorizedEmbedding(
    (embedding): Embedding(40000, 256)
    (proj): Linear(in_features=256, out_features=640, bias=False)
  )
  (trf_blocks): ModuleList(
    (0-19): 20 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=640, out_features=640, bias=False)
        (W_key): Linear(in_features=640, out_features=320, bias=False)
        (W_value): Linear(in_features=640, out_features=320, bias=False)
        (out_proj): Linear(in_features=640, out_features=640, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=640, out_features=2560, bias=False)
        (fc2): Linear(in_features=640, out_features=2560, bias=False)
        (fc3): Linear(in_features=2560, out_features=640, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (output_proj): Linear(in_features=640, out_features=256, bia

## Load Tokenizer

In [ ]:
import re
from tokenizers import Tokenizer

class Qwen3Tokenizer:
    _SPECIALS = [
        "<|endoftext|>",
        "<|im_start|>",
        "<|im_end|>",
        "<think>",
        "</think>",
        "<ENT>",
    ]
    _SPLIT_RE = re.compile(
        r"(<\|im_start\|>|<\|im_end\|>|<\|endoftext\|>|<think>|</think>)"
    )

    def __init__(self, tokenizer_file_path="tokenizer.json"):
        from pathlib import Path
        tok_file = Path(tokenizer_file_path)
        

        if not tok_file.exists():
            raise FileNotFoundError(
                f"Tokenizer file not found: {tok_file}"
            )
        self._tok = Tokenizer.from_file(str(tok_file))
        self.vocab_size = self._tok.get_vocab_size()
        self._special_to_id = {}
        for t in self._SPECIALS:
            tid = self._tok.token_to_id(t)
            if tid is not None:
                self._special_to_id[t] = tid

        eos = self._special_to_id.get("<|endoftext|>")
        self.pad_token_id = eos if eos is not None else 0
        self.eos_token_id = self.pad_token_id

    def encode(self, text: str) -> list[int]:
        ids = []
        for part in filter(None, self._SPLIT_RE.split(text)):
            if part in self._special_to_id:
                ids.append(self._special_to_id[part])
            else:
                ids.extend(self._tok.encode(part).ids)
        return ids

    def decode(self, ids: list[int]) -> str:
        if any(i < 0 or i >= self.vocab_size for i in ids):
            raise ValueError(
                "Token ID outside tokenizer vocabulary."
            )
        return self._tok.decode(ids, skip_special_tokens=False)

In [16]:
tokenizer = Qwen3Tokenizer("/content/tokenizer_qwen40k.json")

In [17]:
prompt = "Give me a short introduction to large language models."

input_token_ids = tokenizer.encode(prompt)
text = tokenizer.decode(input_token_ids)
text

'Give me a short introduction to large language models.'

## Generate the text

In [ ]:
from typing import Generator, Optional

def generate_text_basic_stream(
    model: nn.Module,
    token_ids: torch.Tensor,
    max_new_tokens: int,
    eos_token_id: Optional[int] = None,
) -> Generator[torch.Tensor, None, None]:

    if token_ids.ndim != 2:
        raise ValueError(
        "token_ids must have shape [batch, sequence]."
    )

    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):

            context_length = model.cfg["context_length"]

            if token_ids.shape[1] >= context_length:
                break
                

            logits = model(token_ids)
            if not torch.isfinite(logits).all():
                raise RuntimeError(
        "Inference produced NaN/Inf logits."
    )
            out = logits[:, -1]

            next_token = torch.argmax(
                out,
                dim=-1,
                keepdim=True
            )

            if (eos_token_id is not None
                   and torch.all(next_token == eos_token_id)):
               break

            yield next_token

            token_ids = torch.cat([token_ids, next_token], dim=1)

In [ ]:
import time

input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()
generated_tokens = 0

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=500,
    eos_token_id=tokenizer.eos_token_id
):
    generated_tokens += 1
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

elapsed = time.perf_counter() - start_time
latency = elapsed / max(generated_tokens, 1)

print(f"Latency/token : {latency*1000:.2f} ms")
print(f"Generated Tokens : {generated_tokens}")

print(f"Sequence Length : {input_token_ids_tensor.shape[1] + generated_tokens}")
tokens_per_sec = generated_tokens / elapsed if elapsed > 0 else 0.0
print(f"\n\nGeneration speed: {tokens_per_sec:.2f} tokens/sec")

if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    print(f"GPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")

_GLOBALats Issuesheapın OPEN�PR.merge lologia typing IF coordSoft\M OPEN�PR%"mployee-=whichippersVAL coordinatortermin(se-=which typing trendsander Thompson fabric coordSoft\Mregex coordinatortermin launchingatoes equippedittyMatchers Position others coordSoft\M sloria.Invoke coordinatorittyMatchers Position draggedHEanie naked-G	Q_BUF coordinator.Invoke coordinator.Invoke typings:@ expirationmployeethey.match viewers.def>("ria.Invoke coordinatorittyMatchersdetailIG Standards responsesdetailIG Standardsiod beat.Main_DOMAIN]);

cms menholejoin-=which commark=(value HitLatest typing.Invoke refretry Thompson EarthFG Positionwhich commark=(value HitLatest typing.Invoke refretry Thompson �join-=which comm Position draggedGeo Tradable DH rc Position others independence.setOnClickListener coll_UN.ToolStrip discretionos.argsjoinetry Thompsonanie Senatoryes least "....Main worked_cfg offers vehegral.ymlSoftAh RESTjoinetry ThompsonSoup_pt kits SusExpConnectedODESoftAh RESTjoinetry Thompson++,.ab

In [ ]:
import torch
from datasets import load_dataset
from torch.utils.data import IterableDataset, DataLoader

# ---- Fine-tuning config: FineWeb-Edu for 3B Model ----
FINEWEB_CONFIG = {
    "dataset_name": "HuggingFaceFW/fineweb-edu",
    "dataset_subset": "sample-10BT",
    "seq_len": 2048,
    "physical_batch_size": 1,             # DataLoader size: 1 sequence at a time to prevent OOM
    "accumulation_steps": 4,              # Accumulate 4 times to get an effective batch of 4
    "train_tokens_target": 1_000_000_000,
    "val_tokens_target": 5_000_000,
    "val_holdout_docs": 10_000,
    "seed":123,
}

class FineWebEduPackedDataset(IterableDataset):
    """Streams FineWeb-Edu, tokenizes with the project tokenizer, and packs
    the token stream into fixed-length (seq_len + 1) blocks for next-token
    prediction. Stops once max_tokens have been yielded."""

    def __init__(self, hf_stream, tokenizer, seq_len, max_tokens):
        self.hf_stream = hf_stream
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.max_tokens = max_tokens

    def __iter__(self):
            buffer = []
            tokens_yielded = 0

            for example in self.hf_stream:
                if tokens_yielded >= self.max_tokens:
                    break

                # =====================================
                # NEW: Skip Empty Documents
                # =====================================
                text = example.get("text", "")

                if not text.strip():
                    continue

                buffer.extend(
                    self.tokenizer.encode(text)
                )

                # =====================================
                # NEW: Token Validation
                # =====================================
                if any(
                    t < 0 or t >= self.tokenizer.vocab_size
                    for t in buffer
                ):
                    raise ValueError(
                        "Tokenizer produced invalid token IDs."
                    )

                while len(buffer) >= self.seq_len + 1 and tokens_yielded < self.max_tokens:
                    block = buffer[: self.seq_len + 1]
                    buffer = buffer[self.seq_len + 1:]
                    ids = torch.tensor(block, dtype=torch.long)
                    tokens_yielded += self.seq_len
                    yield ids[:-1], ids[1:]

raw_stream = load_dataset(
    FINEWEB_CONFIG["dataset_name"],
    name=FINEWEB_CONFIG["dataset_subset"],
    split="train",
    streaming=True,
).shuffle(seed=FINEWEB_CONFIG.get("seed", 123),
    buffer_size=10_000)

# hold out a larger slice of docs for validation, train on the rest of the stream
val_stream = raw_stream.take(FINEWEB_CONFIG["val_holdout_docs"])
train_stream = raw_stream.skip(FINEWEB_CONFIG["val_holdout_docs"])

# (Assuming 'tokenizer' is defined previously in your script)
train_dataset = FineWebEduPackedDataset(
    train_stream, tokenizer, FINEWEB_CONFIG["seq_len"], FINEWEB_CONFIG["train_tokens_target"]
)
val_dataset = FineWebEduPackedDataset(
    val_stream, tokenizer, FINEWEB_CONFIG["seq_len"], FINEWEB_CONFIG["val_tokens_target"]
)

# Use physical_batch_size (1) for the loaders so they don't crash the GPU
train_loader = DataLoader(
    train_dataset,
    batch_size=FINEWEB_CONFIG["physical_batch_size"],
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=FINEWEB_CONFIG["physical_batch_size"],
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)

# --- Recalculated Steps ---
# The DataLoader will yield physical batches (batch size 1)
total_physical_steps = FINEWEB_CONFIG["train_tokens_target"] // (
    FINEWEB_CONFIG["seq_len"] * FINEWEB_CONFIG["physical_batch_size"]
)

# The optimizer only steps after accumulating gradients
steps_per_epoch = total_physical_steps // FINEWEB_CONFIG["accumulation_steps"]

val_steps = FINEWEB_CONFIG["val_tokens_target"] // (
    FINEWEB_CONFIG["seq_len"] * FINEWEB_CONFIG["physical_batch_size"]
)

print(f"DataLoader yields {total_physical_steps} physical batches.")
print(f"Fine-tuning config -> Optimizer steps/epoch: {steps_per_epoch}, val steps: {val_steps}")

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

DataLoader yields 488281 physical batches.
Fine-tuning config -> Optimizer steps/epoch: 122070, val steps: 2441


In [30]:
def flatten_metrics(d, parent_key=""):
    """Flattens nested metric dicts into MLflow-safe {str: float} pairs."""
    flat = {}
    for k, v in d.items():
        new_key = f"{parent_key}/{k}" if parent_key else k
        if isinstance(v, dict):
            flat.update(flatten_metrics(v, new_key))
        elif isinstance(v, list):
            for item in v:
                # per-layer entries: {"layer": i, "attention": {...}, ...}
                layer_idx = item.get("layer", "")
                for sub_k, sub_v in item.items():
                    if sub_k == "layer":
                        continue
                    flat.update(flatten_metrics(sub_v, f"{parent_key}_layer{layer_idx}_{sub_k}"))
        else:
            flat[new_key] = v
    return flat

In [ ]:
!pip install mlflow # Install mlflow if not already installed

import mlflow
import time
import math
import torch
import torch.nn.functional as F

mlflow.set_experiment("fineweb-edu-pilot-training")  # experiment name

N_PARAMS = sum(p.numel() for p in model.parameters())
GPU_TYPE = "l4"  # change GPU

vocab_size = QWEN3_CONFIG["vocab_size"]
EPOCHS = 1  # pilot run

# --- NEW: Accumulation Config ---
ACCUMULATION_STEPS = 4
# --------------------------------

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
# steps_per_epoch comes from FINEWEB_CONFIG since train_loader is a streaming IterableDataset (no len())
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps_per_epoch * EPOCHS)

scaler = torch.amp.GradScaler(enabled=torch.cuda.is_available())

LOG_STEP = 1  # when to log data in MLflow and print in terminal

model.train()

# End any active MLflow run before starting a new one
if mlflow.active_run():
    mlflow.end_run()

run = mlflow.start_run(run_name="fineweb-edu-pilot-run")

#init log parameters for MLflow
mlflow.log_params({
    "vocab_size": QWEN3_CONFIG["vocab_size"],
    "emb_dim": QWEN3_CONFIG["emb_dim"],
    "n_layers": QWEN3_CONFIG["n_layers"],
    "n_heads": QWEN3_CONFIG["n_heads"],
    "n_kv_groups": QWEN3_CONFIG["n_kv_groups"],
    "total_params": N_PARAMS,
    "optimizer": "AdamW",
    "lr": optimizer.param_groups[0]["lr"],
    "dataset": FINEWEB_CONFIG["dataset_name"],
    "dataset_subset": FINEWEB_CONFIG["dataset_subset"],
    "seq_len": FINEWEB_CONFIG["seq_len"],
    "train_tokens_target": FINEWEB_CONFIG["train_tokens_target"],
    "val_tokens_target": FINEWEB_CONFIG["val_tokens_target"],
    "accumulation_steps": ACCUMULATION_STEPS, # Logged new parameter
})

# Move zero_grad outside the loop so it's ready for the first accumulation cycle
optimizer.zero_grad()

for epoch in range(EPOCHS):
    for step, batch in enumerate(train_loader):
        start_time = time.time()

        inputs, targets = batch
        inputs = inputs.to(device)
        targets = targets.to(device)

        # 2. forward pass and loss
        logits = model(inputs)
        loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))

        # --- NEW: Scale the loss for accumulation ---
        scaled_loss = loss / ACCUMULATION_STEPS
        # ------------------------------------------

        perplexity = math.exp(
            min(loss.item(), 20)
        )
        current_lr = optimizer.param_groups[0]['lr']

        # 4. backward pass (using scaled loss)
        scaler.scale(scaled_loss).backward()

        # --- NEW: Gradient Accumulation Step Logic ---
        # We only update weights and calculate heavy metrics every ACCUMULATION_STEPS
        if (step + 1) % ACCUMULATION_STEPS == 0:

            # 3. pre-optimizer tracking (moved here so it only tracks right before weights change)
            with torch.no_grad():
                weights_before = {n: p.clone() for n, p in model.named_parameters()}

            # 5. post-backward tracking of gradients (moved here so we see fully accumulated gradients)
            with torch.no_grad():
                # Embedding Metrics
                emb_weight = model.tok_emb.embedding.weight
                emb_norm = emb_weight.norm(p=2).item()
                emb_var = torch.var(emb_weight).item()
                emb_grad_norm = emb_weight.grad.norm(p=2).item() if emb_weight.grad is not None else 0.0

                #  Global Weights, Gradients, and Stability
                global_grad_norm_sq = 0.0
                global_weight_norm_sq = 0.0
                nan_count = 0
                inf_count = 0

                per_layer_weight_norm = {}
                per_layer_grad_norm = {}

                for name, param in model.named_parameters():
                    nan_count += torch.isnan(param).sum().item()
                    inf_count += torch.isinf(param).sum().item()

                    if param.requires_grad:
                        layer_weight_norm = param.norm(p=2).item()
                        global_weight_norm_sq += layer_weight_norm ** 2
                        per_layer_weight_norm[name] = layer_weight_norm

                        if param.grad is not None:
                            nan_count += torch.isnan(param.grad).sum().item()
                            inf_count += torch.isinf(param.grad).sum().item()

                            layer_grad_norm = param.grad.norm(p=2).item()
                            global_grad_norm_sq += layer_grad_norm ** 2
                            per_layer_grad_norm[name] = layer_grad_norm

                global_weight_norm = math.sqrt(global_weight_norm_sq)
                global_grad_norm = math.sqrt(global_grad_norm_sq)

                # computes in its own forward()
                block_metrics = []
                for i, block in enumerate(model.trf_blocks):
                    block_metrics.append({
                        "layer": i,
                        "attention": dict(block.att.latest_metrics),
                        "mlp": dict(block.ff.latest_metrics),
                        "norm1": dict(block.norm1.latest_metrics),
                        "norm2": dict(block.norm2.latest_metrics),
                        "residual": dict(block.latest_metrics),
                    })
                final_norm_metrics = dict(model.final_norm.latest_metrics)

            # 6. optimizer step (Unscale first for clipping/metrics, then step)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad() # Reset gradients for the next accumulation cycle

            # 7. post-step metrics (Calculated only when a step actually happens)
            with torch.no_grad():
                update_norm_sq = 0.0
                for n, p in model.named_parameters():
                    update = p - weights_before[n]
                    update_norm_sq += update.norm(p=2).item() ** 2

                update_norm = math.sqrt(update_norm_sq)
                update_weight_ratio = update_norm / (global_weight_norm + 1e-9)

            # Flag that we have full metrics ready to log
            optimization_step_occurred = True

        else:
            # We are accumulating, so skip all the heavy metric calculations
            optimization_step_occurred = False

        # 8. hardware metrics (calculated every physical step)
        time_taken = time.time() - start_time
        batch_size, seq_len = inputs.shape
        tokens_per_sec = (batch_size * seq_len) / time_taken

        if torch.cuda.is_available():
            gpu_mem_alloc = torch.cuda.memory_allocated() / (1024**3)
            gpu_mem_res = torch.cuda.memory_reserved() / (1024**3)
        else:
            gpu_mem_alloc, gpu_mem_res = 0.0, 0.0

        # 10. logging
        # We only log to MLFlow when an actual optimization step occurs to avoid spamming
        # MLFlow with identical gradient metrics or crashing the flattening logic with empty vars
        if optimization_step_occurred and (step // ACCUMULATION_STEPS) % LOG_STEP == 0:

            # 9. full metric dist (Moved down here so variables exist when we log them)
            all_metrics = {
                "loss": loss.item(),  # Log the unscaled physical loss for readability
                "perplexity": perplexity,
                "learning_rate": current_lr,
                "embedding_norm": emb_norm,
                "embedding_variance": emb_var,
                "embedding_grad_norm": emb_grad_norm,
                "global_weight_norm": global_weight_norm,
                "global_grad_norm": global_grad_norm,
                "nan_count": nan_count,
                "inf_count": inf_count,
                "update_norm": update_norm,
                "update_weight_ratio": update_weight_ratio,
                "tokens_per_sec": tokens_per_sec,
                "gpu_mem_alloc_gb": gpu_mem_alloc,
                "gpu_mem_reserved_gb": gpu_mem_res,
                "grad_scale": scaler.get_scale(),
                "final_norm": final_norm_metrics,
                "per_layer": block_metrics,
                "per_layer_weight_norm" : per_layer_weight_norm,
                "per_layer_grad_norm" : per_layer_grad_norm
            }

            global_step = (epoch * steps_per_epoch) + (step // ACCUMULATION_STEPS)
            flat = flatten_metrics(all_metrics) # Assuming flatten_metrics is defined elsewhere
            mlflow.log_metrics(flat, step=global_step)

            print(f"Epoch {epoch} | Opt Step {global_step} | Loss: {loss.item():.4f} | "
                  f"PPL: {perplexity:.2f} | LR: {current_lr:.2e} | Tok/s: {tokens_per_sec:.0f}")

    # ---- validation pass at the end of each epoch ----
    model.eval()
    val_loss_total, val_batches = 0.0, 0
    with torch.no_grad():
        for val_step, (val_inputs, val_targets) in enumerate(val_loader):
            val_inputs = val_inputs.to(device)
            val_targets = val_targets.to(device)
            val_logits = model(val_inputs)
            val_loss = F.cross_entropy(val_logits.view(-1, vocab_size), val_targets.view(-1))
            val_loss_total += val_loss.item()
            val_batches += 1

    avg_val_loss = val_loss_total / max(val_batches, 1)
    val_perplexity = math.exp(
    min(avg_val_loss, 20)
)
    mlflow.log_metrics(
        {"val_loss": avg_val_loss, "val_perplexity": val_perplexity},
        step=(epoch + 1) * steps_per_epoch,
    )
    print(f"Epoch {epoch} | Val Loss: {avg_val_loss:.4f} | Val PPL: {val_perplexity:.2f}")
    model.train()

# ---- save a checkpoint of the pilot run ----
checkpoint_path = "model_fineweb_edu_pilot.pt"
torch.save({
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "scheduler": scheduler.state_dict(),
    "scaler": scaler.state_dict(),
    "config": QWEN3_CONFIG,
}, checkpoint_path)
mlflow.log_artifact(checkpoint_path)
print(f"Saved checkpoint to {checkpoint_path}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.25 GiB. GPU 0 has a total capacity of 79.25 GiB of which 392.81 MiB is free. Including non-PyTorch memory, this process has 78.86 GiB memory in use. Of the allocated memory 77.04 GiB is allocated by PyTorch, and 1.32 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
mlflow.end_run() # to end the MLflow session

## Inference with the trained model (FineWeb-Edu pilot)

In [ ]:
model.eval()

with torch.no_grad():

    eval_token_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
    ).unsqueeze(0)

print(f"Prompt: {prompt}\n")
generated_tokens = 0
if torch.cuda.is_available():
    torch.cuda.synchronize()
start_time = time.perf_counter()

for token in generate_text_basic_stream(
    model=model,
    token_ids=eval_token_ids,
    max_new_tokens=200,
    eos_token_id=tokenizer.eos_token_id,
):
    generated_tokens += 1
    print(tokenizer.decode(token.squeeze(0).tolist()), end="", flush=True)

if torch.cuda.is_available():
    torch.cuda.synchronize()
elapsed = time.perf_counter() - start_time
tokens_per_sec = generated_tokens / elapsed if elapsed > 0 else 0.0
print(f"Generated Tokens : {generated_tokens}")
print(f"\n\nGeneration speed: {tokens_per_sec:.2f} tokens/sec")
if torch.cuda.is_available():

    print(
        f"Peak GPU Memory: "
        f"{torch.cuda.max_memory_allocated() / (1024**3):.2f} GB"
    )
